# Ejemplo 2 · Algoritmos genéticos: mejora de parámetros

**Propósito:** observar cómo la selección, el tamaño de población, las generaciones y la mutación influyen en la convergencia.

**No se evaluará programación.** El código está completo; solamente modifica el número de lista. Luego ejecuta las celdas en orden, interpreta las gráficas y completa las preguntas.

**Reglas para comparar:** todos los experimentos usan la misma función individual, el mismo cruce **aritmético**, mutación **gaussiana** y elitismo de 1. Se comparan **torneo** y **ruleta**. *Menor valor de f(x) significa mejor solución.*


In [ ]:
# ÚNICA MODIFICACIÓN DEL ALUMNO EN EL CUADERNO GUIADO
NUMERO_LISTA = 12  # Sustituye 12 por tu número de lista (1 a 100)


## 1. Código completo del algoritmo (no modificar)
Ejecuta esta celda. Solo se muestra para que sepas qué hace el programa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# FUNCIÓN OBJETIVO: minimizar f(x) = (x - a)^2
# El número de lista cambia a, pero conserva la dificultad del problema.
if not isinstance(NUMERO_LISTA, int) or not (1 <= NUMERO_LISTA <= 100):
    raise ValueError("NUMERO_LISTA debe ser un entero del 1 al 100")
a = 4.0 + 0.13 * NUMERO_LISTA
LIM_INF, LIM_SUP = -4.0, 21.0

def objetivo(x):
    return (np.asarray(x) - a) ** 2

# Rueda probabilística: como el problema es de MINIMIZACIÓN,
# transformamos el costo en una aptitud positiva: 1 / (1 + f(x)).
def elegir_padre(poblacion, costos, metodo, rng):
    if metodo == "torneo":
        i, j = rng.choice(len(poblacion), size=2, replace=False)
        return poblacion[i] if costos[i] <= costos[j] else poblacion[j]
    if metodo == "ruleta":
        pesos = 1.0 / (1.0 + costos)
        probabilidades = pesos / pesos.sum()
        return poblacion[rng.choice(len(poblacion), p=probabilidades)]
    raise ValueError("Selecciona 'torneo' o 'ruleta'")

def algoritmo_genetico(config, semilla, guardar_poblaciones=False):
    n = int(config["poblacion"])
    generaciones = int(config["generaciones"])
    pm = float(config["probabilidad_mutacion"])
    sigma = float(config["desviacion_mutacion"])
    elite = int(config["elitismo"])
    metodo = config["seleccion"]
    if not (4 <= n <= 100 and 1 <= generaciones <= 300 and 0 <= pm <= 1 and
            0 <= sigma <= 5 and 0 <= elite < n and metodo in ["torneo", "ruleta"]):
        raise ValueError("Configuración fuera de rango; revisa la tabla de parámetros")
    rng = np.random.default_rng(semilla)
    poblacion = rng.uniform(LIM_INF, LIM_SUP, size=n)
    historial = []
    poblaciones = []
    mejor_global = float("inf")
    mejor_x = None
    # En cada generación se evalúa la población completa.
    # Se registra la generación inicial (0) y las generaciones nuevas.
    for g in range(generaciones + 1):
        costos = objetivo(poblacion)
        i_mejor = int(np.argmin(costos))
        if float(costos[i_mejor]) < mejor_global:
            mejor_global = float(costos[i_mejor])
            mejor_x = float(poblacion[i_mejor])
        historial.append({"generacion": g,
                          "mejor_generacion": float(costos[i_mejor]),
                          "mejor_acumulado": mejor_global,
                          "promedio": float(np.mean(costos)),
                          "diversidad": float(np.std(poblacion))})
        if guardar_poblaciones:
            poblaciones.append(poblacion.copy())
        if g == generaciones:
            break
        # Elitismo: conservar los mejores individuos, sin modificarlos.
        indices = np.argsort(costos)[:elite]
        nueva = list(poblacion[indices].copy())
        while len(nueva) < n:
            p1 = elegir_padre(poblacion, costos, metodo, rng)
            p2 = elegir_padre(poblacion, costos, metodo, rng)
            # Cruce aritmético: x_hijo = alfa*p1 + (1-alfa)*p2.
            alfa = rng.uniform(0, 1)
            hijo = alfa*p1 + (1-alfa)*p2
            if rng.random() < pm:
                hijo += rng.normal(0, sigma)  # Mutación gaussiana
            nueva.append(float(np.clip(hijo, LIM_INF, LIM_SUP)))
        poblacion = np.asarray(nueva)
    return {"historial": pd.DataFrame(historial), "mejor_x": mejor_x,
            "mejor_f": mejor_global, "poblaciones": poblaciones,
            "evaluaciones": n*(generaciones+1)}

def comparar_experimentos(experimentos, repeticiones=8):
    resultados = {}
    filas = []
    # Repetimos con las mismas semillas en todos los experimentos.
    semillas = [NUMERO_LISTA*1000 + 31*i for i in range(repeticiones)]
    for nombre, cfg in experimentos.items():
        corridas = [algoritmo_genetico(cfg, s) for s in semillas]
        resultados[nombre] = corridas
        valores = np.array([r["mejor_f"] for r in corridas])
        filas.append({"Experimento": nombre, "Selección": cfg["seleccion"],
                      "Población": cfg["poblacion"], "Generaciones": cfg["generaciones"],
                      "P_mutación": cfg["probabilidad_mutacion"],
                      "σ_mutación": cfg["desviacion_mutacion"], "Elitismo": cfg["elitismo"],
                      "Error medio final": round(float(valores.mean()), 6),
                      "Error peor corrida": round(float(valores.max()), 6),
                      "Evaluaciones por corrida": corridas[0]["evaluaciones"]})
    return pd.DataFrame(filas), resultados

def graficar_comparacion(resultados, titulo="Comparación de convergencia"):
    fig, axes = plt.subplots(1, 2, figsize=(12,4.1))
    for nombre, corridas in resultados.items():
        # Todos los ensayos del mismo experimento tienen la misma longitud.
        curvas = np.stack([r["historial"]["mejor_acumulado"].to_numpy() for r in corridas])
        x = corridas[0]["historial"]["generacion"].to_numpy()
        mediana = np.median(curvas, axis=0)
        axes[0].plot(x, mediana, label=nombre)
        axes[1].plot(x, mediana, label=nombre)
    axes[0].set_title("Mediana del mejor error acumulado")
    axes[1].set_title("Detalle de valores pequeños (escala log)")
    axes[1].set_yscale("log")
    for ax in axes:
        ax.set_xlabel("Generación")
        ax.set_ylabel("f(x) = error (menor es mejor)")
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

def graficar_poblacion(config, semilla):
    r = algoritmo_genetico(config, semilla, guardar_poblaciones=True)
    xs = np.linspace(LIM_INF, LIM_SUP, 500)
    mostrar = sorted(set([0, 1, 3, config["generaciones"] // 2, config["generaciones"]]))
    fig, axs = plt.subplots(1, len(mostrar), figsize=(3.2*len(mostrar),3), sharey=True)
    for ax,g in zip(np.atleast_1d(axs),mostrar):
        ax.plot(xs, objetivo(xs), lw=1.6)
        pob = r["poblaciones"][g]
        ax.scatter(pob, objetivo(pob), s=22, zorder=3)
        ax.set_title(f"Generación {g}")
        ax.set_xlabel("x")
        ax.grid(alpha=.25)
    np.atleast_1d(axs)[0].set_ylabel("f(x)")
    plt.tight_layout()
    plt.show()
    return r

print(f"Número de lista: {NUMERO_LISTA} | Función individual: f(x) = (x - {a:.2f})²")
print(f"Mínimo matemático conocido: x = {a:.2f}, f(x) = 0")


## Antes de comparar: ¿qué significan las variables?

| Variable | Significado | Ejemplo |
|---|---|---|
| `seleccion` | Forma de elegir padres: `torneo` o `ruleta` | Torneo entre dos individuos o ruleta de aptitudes |
| `poblacion` | Número de soluciones simultáneas | 8 individuos |
| `generaciones` | Número de ciclos de reproducción | 12 generaciones nuevas |
| `probabilidad_mutacion` | Probabilidad de mutar **cada hijo** | 0.30 = 30% |
| `desviacion_mutacion` | Tamaño habitual de una perturbación gaussiana | σ = 0.38 |
| `elitismo` | Mejores individuos que pasan intactos a la nueva población | 1 |

La **ruleta** asigna mayores probabilidades a los valores de menor error usando `aptitud = 1/(1+f(x))`; la selección por **torneo** compara dos candidatos aleatorios y elige al que tiene menor error. Ambas son adecuadas para esta función.

**Importante:** la parábola se desplaza según el número de lista. El mínimo real se conoce porque es una práctica didáctica, pero el algoritmo no recibe ese valor para construir sus soluciones.


## 2. Graficar la función objetivo

In [ ]:
x = np.linspace(LIM_INF, LIM_SUP, 500)
plt.figure(figsize=(8,3.5))
plt.plot(x, objetivo(x), label="Función objetivo")
plt.scatter([a], [0], marker="x", s=90, label="Mínimo matemático")
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("La función que resolverá el algoritmo genético")
plt.grid(alpha=.3); plt.legend(); plt.show()


## 3. Experimentos: modifica los valores

Todos los experimentos parten de **BASE**. En la celda siguiente cambia **solamente el valor señalado** de cada experimento (no cambies el resto):

| Experimento | Instrucción de cambio frente a BASE |
|---|---|
| E1 | No cambiar nada: configuración inicial |
| E2 | `seleccion = "ruleta"` |
| E3 | `poblacion = 20` |
| E4 | `generaciones = 35` |
| E5 | `probabilidad_mutacion = 0.70` |
| E6 | `desviacion_mutacion = 1.20` |

**Primero:** ejecuta con todos los valores de BASE y observa que se obtienen curvas iguales en los seis experimentos. **Después:** realiza las seis modificaciones indicadas y vuelve a ejecutar. Así identificaremos el efecto de cada cambio.


In [ ]:
BASE = {"seleccion":"torneo", "poblacion":8, "generaciones":12,
        "probabilidad_mutacion":0.30, "desviacion_mutacion":0.38,
        "elitismo":1}

# CAMBIA UN SOLO VALOR EN CADA EXPERIMENTO CON LAS INSTRUCCIONES DEL PROFESOR.
# Todos inician iguales para que el estudiante realice los cambios.
E1 = dict(BASE)
E2 = dict(BASE, seleccion="torneo")                  # Cambiar a "ruleta"
E3 = dict(BASE, poblacion=8)                         # Cambiar a 20
E4 = dict(BASE, generaciones=12)                     # Cambiar a 35
E5 = dict(BASE, probabilidad_mutacion=0.30)          # Cambiar a 0.70
E6 = dict(BASE, desviacion_mutacion=0.38)           # Cambiar a 1.20

experimentos = {
    "E1 · Base": E1,
    "E2 · Selección": E2,
    "E3 · Población": E3,
    "E4 · Generaciones": E4,
    "E5 · Frecuencia mutación": E5,
    "E6 · Tamaño mutación": E6,
}
resumen, resultados = comparar_experimentos(experimentos, repeticiones=8)
display(resumen)
graficar_comparacion(resultados, "Experimentos guiados · mismo número de lista")


## 4. Visualizar cómo se mueve la población

In [ ]:
# Observa individuos de varias generaciones con la CONFIGURACIÓN BASE.
resultado_base = graficar_poblacion(BASE, semilla=NUMERO_LISTA*1000)
print("Mejor x obtenido:", round(resultado_base["mejor_x"],5))
print("Mejor f(x) obtenido:", round(resultado_base["mejor_f"],7))


## Hoja de observaciones guiadas (responder en el cuaderno o libreta)

**A. Comprensión de gráficas**
1. ¿Qué representa el eje horizontal de la gráfica de convergencia? ¿Y el vertical?
2. ¿Por qué una curva más baja representa un resultado mejor en este problema?
3. Si una curva permanece horizontal durante varias generaciones, ¿demuestra que encontró el óptimo global? Explica.

**B. Comparar experimentos (todos respecto de E1)**
4. E1 frente a E2: ¿cuál selección consiguió menor **error medio final** en tus ocho repeticiones? ¿Cambió mucho?
5. E1 frente a E3: ¿qué sucedió al aumentar la población de 8 a 20? ¿Cuántas evaluaciones adicionales se realizaron?
6. E1 frente a E4: ¿qué sucedió al aumentar las generaciones de 12 a 35? ¿Cuál fue su costo en evaluaciones?
7. E1 frente a E5 y E6: ¿qué diferencias observaste entre **mutar más veces** y **mutar con cambios más grandes**?

**C. Cierre**
8. Con base en la tabla, ¿qué experimento escogerías para obtener buen resultado sin realizar demasiadas evaluaciones? Justifica usando **dos números** de la tabla.

**Nota metodológica:** las gráficas comparan la **mediana** del mejor error acumulado de ocho ejecuciones. La tabla muestra el **promedio** del error final y su peor caso. Las evaluaciones se calculan como `población × (generaciones + 1)`; permiten aproximar el esfuerzo computacional. **No se deben atribuir diferencias al método si son muy pequeñas o inestables.**
